In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
zarif98sjs_gait_in_parkinsons_disease_path = kagglehub.dataset_download('zarif98sjs/gait-in-parkinsons-disease')

print('Data source import complete.')


100%|██████████| 86.5M/86.5M [00:00<00:00, 194MB/s]

Extracting files...


Data source import complete.


In [2]:
##pip install numpy==1.22.0

In [3]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import tensorflow as tf
import random

import os


In [5]:
input_dir = zarif98sjs_gait_in_parkinsons_disease_path
len(os.listdir(input_dir))

309

In [6]:
features = ['Time', 'L1' , 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'Total_Force_Left', 'Total_Force_Right']
!mkdir CSV

In [7]:
i = 0
for file in os.listdir(input_dir):
    i = i+1
    ##print(i)
    if file[0:2]=='Ga' or file[0:2]=='Ju' or file[0:2] == 'Si':
        pass
        ##print(file)
        df = pd.read_csv(input_dir + '/' + file, header = None, sep = '\t', on_bad_lines = 'skip')

        df.columns = features

        name = 'CSV/' + file.split('.')[0] + '.csv'
        df.to_csv(name, index = None)

print(i)

309


In [8]:
demographics = pd.read_csv(input_dir + '/demographics.txt', delim_whitespace = True)
sub_id = demographics.ID.to_list()
sub_names = []
for name in os.listdir('CSV'):
    sub_name = name.split('_')[0]
    sub_names.append(sub_name)

print("Subjects Count = ", len(sub_id))
print("Files Count = ", len(sub_names))

Subjects Count =  166
Files Count =  306


/tmp/ipykernel_4960/1169665948.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  demographics = pd.read_csv(input_dir + '/demographics.txt', delim_whitespace = True)


In [9]:
!mkdir train
!mkdir validate
!mkdir test

In [10]:
count = len(os.listdir('CSV'))
train_count = int(70/100*count)
valid_count = int(15/100*count)
test_count = count - train_count - valid_count

print("Training count = ", train_count)
print("Validate count = ", valid_count)
print("Test count = ", test_count)

Training count =  214
Validate count =  45
Test count =  47


In [11]:
def load():
    count = len(os.listdir('CSV'))
    train_count = int(70/100*count)
    valid_count = int(15/100*count)
    test_count = count - train_count - valid_count

    train_X = list()
    train_Y = list()

    valid_X = list()
    valid_Y = list()

    test_X = list()
    test_Y = list()

    i = 0

    for name in os.listdir('CSV'):
        ##if(i>10):
        ##    break
        ##print(name)
        sub_id = name.split('_')[0]
        sub_class = demographics[demographics['ID']==sub_id]['Group']
        sub_class -= 1
        sub_class.to_string(index=False).strip()
        ##(sub_class)

        dataframe = pd.read_csv('CSV/' + name)
        full_size = 400
        skip = 50

        X = list()
        ##Y = list()
        ##Y.append(sub_class)

        max_j = 150
        for j in range(0, dataframe.shape[0], skip):
            if dataframe.shape[0]>=full_size+j:
                temp = list()
                for feature in features:
                    temp.append(dataframe.iloc[j:j+full_size, ][feature].to_numpy())
                ##X.append(temp)

            if(i<train_count):
                train_X.append(temp)
                train_Y.append(sub_class)
                        ##print(i, name, "train")
            elif(train_count<=i<valid_count+train_count):
                valid_X.append(temp)
                valid_Y.append(sub_class)
                        ##print(i, name, "valid")
            else:
                test_X.append(temp)
                test_Y.append(sub_class)
                    ##print(i, name, "test")
        print(i, name)
        i+=1

    train_X = np.array(train_X)
    train_Y = tf.keras.utils.to_categorical(train_Y)

    valid_X = np.array(valid_X)
    valid_Y = tf.keras.utils.to_categorical(valid_Y)

    test_X = np.array(test_X)
    test_Y = tf.keras.utils.to_categorical(test_Y)

    """valid_X = np.destack(valid_X)
    valid_Y = np.destack(valid_Y)

    test_X = np.destack(test_X)
    test_Y = np.destack(test_Y)"""

    return train_X, train_Y, valid_X, valid_Y, test_X, test_Y

In [12]:
train_X, train_Y, valid_X, valid_Y, test_X, test_Y = load()
print("Training Data = ", train_X.shape)
print("Training Class = ", train_Y.shape)
print("Valid Data = ", valid_X.shape)
print("Valid Class = ", valid_Y.shape)
print("Test Data = ", test_X.shape)
print("Test Class = ", test_Y.shape)

0 JuPt15_01.csv
1 JuPt09_03.csv
2 JuPt03_01.csv
3 GaPt16_10.csv
4 JuPt21_03.csv
5 GaCo02_02.csv
6 JuPt11_04.csv
7 SiPt08_01.csv
8 JuPt01_03.csv
9 JuPt21_02.csv
10 JuPt01_02.csv
11 JuPt15_05.csv
12 JuPt23_05.csv
13 JuCo15_01.csv
14 JuPt26_06.csv
15 SiCo16_01.csv
16 GaPt21_02.csv
17 GaPt09_02.csv
18 SiCo07_01.csv
19 SiCo24_01.csv
20 JuPt20_07.csv
21 SiCo12_01.csv
22 GaPt12_02.csv
23 GaPt32_10.csv
24 GaPt26_10.csv
25 GaPt25_01.csv
26 SiPt19_01.csv
27 GaPt30_02.csv
28 JuPt29_01.csv
29 SiPt30_01.csv
30 JuCo17_01.csv
31 JuPt26_05.csv
32 JuPt06_05.csv
33 SiPt20_01.csv
34 JuPt11_07.csv
35 JuCo11_01.csv
36 GaPt19_10.csv
37 JuPt20_05.csv
38 SiCo14_01.csv
39 GaCo04_02.csv
40 GaPt20_10.csv
41 JuPt06_07.csv
42 JuPt11_02.csv
43 SiCo26_01.csv
44 JuPt19_01.csv
45 SiCo08_01.csv
46 SiPt27_01.csv
47 GaPt30_01.csv
48 GaCo05_02.csv
49 SiPt24_01.csv
50 SiPt09_01.csv
51 GaPt20_02.csv
52 JuPt10_06.csv
53 JuPt28_04.csv
54 GaPt16_02.csv
55 SiPt18_01.csv
56 GaPt32_01.csv
57 GaCo14_01.csv
58 JuPt01_01.csv
59 JuPt

In [13]:
train_X.reshape(train_X.shape[0], 19, train_X.shape[2], 1)
valid_X.reshape(valid_X.shape[0], 19, valid_X.shape[2], 1)
test_X.reshape(test_X.shape[0], 19, test_X.shape[2], 1)

array([[[[0.00000e+00],
         [1.00000e-02],
         [2.00000e-02],
         ...,
         [3.96970e+00],
         [3.97970e+00],
         [3.98970e+00]],

        [[2.60700e+01],
         [2.60700e+01],
         [2.60700e+01],
         ...,
         [1.31230e+02],
         [1.36620e+02],
         [1.36620e+02]],

        [[1.23750e+02],
         [1.23750e+02],
         [1.23750e+02],
         ...,
         [2.15600e+02],
         [2.30560e+02],
         [2.53770e+02]],

        ...,

        [[3.17900e+01],
         [3.17900e+01],
         [2.91500e+01],
         ...,
         [7.66700e+01],
         [7.66700e+01],
         [7.66700e+01]],

        [[6.81890e+02],
         [6.81890e+02],
         [6.82660e+02],
         ...,
         [6.72320e+02],
         [7.61310e+02],
         [8.40620e+02]],

        [[6.89920e+02],
         [6.89920e+02],
         [6.79250e+02],
         ...,
         [8.40510e+02],
         [7.81220e+02],
         [7.11480e+02]]],


       [[[5.00000e-01],


In [14]:
##pip install keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, Dropout

In [15]:
model = Sequential()

model.add(Conv2D(filters=32, kernel_size=3, padding='same', activation='relu', input_shape = (19, 400, 1)))
model.add(Conv2D(filters = 32, kernel_size = 3, padding = 'same', activation = 'relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Dropout(0.5))

model.add(Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'))
model.add(Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Dropout(0.5))

#------------------------------------
# Conv Block 3: 64 Filters, MaxPool.
#------------------------------------
model.add(Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'))
model.add(Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

#------------------------------------
# Flatten the convolutional features.
#------------------------------------
model.add(Flatten())
model.add(Dense(128, activation='elu'))
model.add(Dense(2, activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 19, 400, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 19, 400, 32)    │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 9, 200, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 9, 200, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 200, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 9, 200, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 4, 100, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4, 100, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 4, 100, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 100, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 2, 50, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       819,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 958,434 (3.66 MB)

 Trainable params: 958,434 (3.66 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
from keras.optimizers import Adam

model.compile(optimizer=Adam(learning_rate = 0.001),
              loss = 'binary_crossentropy',
              metrics = ['accuracy',
                         tf.keras.metrics.Precision(name = 'precision'),
                         tf.keras.metrics.TruePositives(name ='tp'),
                         tf.keras.metrics.FalseNegatives(name = 'fn'),
                         tf.keras.metrics.FalsePositives(name = 'fp'),
                         tf.keras.metrics.TrueNegatives(name = 'tn'),
                         tf.keras.metrics.Recall(name='recall'),
                         tf.keras.metrics.AUC(name='auc')])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 19, 400, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 19, 400, 32)    │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 9, 200, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 9, 200, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 200, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 9, 200, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 4, 100, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4, 100, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 4, 100, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 100, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 2, 50, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       819,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 958,434 (3.66 MB)

 Trainable params: 958,434 (3.66 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
from keras import callbacks
early_stop = callbacks.EarlyStopping(monitor = "accuracy", mode = "auto", restore_best_weights = True)

In [19]:
history = model.fit(train_X,
                   train_Y,
                   batch_size = 32,
                   epochs = 3,
                   verbose = 1,
                   validation_data = (valid_X, valid_Y),
                   callbacks = [early_stop])

Epoch 1/3
1462/1462 ━━━━━━━━━━━━━━━━━━━━ 1160s 791ms/step - accuracy: 0.8440 - auc: 0.9245 - fn: 7296.0000 - fp: 7296.0000 - loss: 0.4461 - precision: 0.8440 - recall: 0.8440 - tn: 39460.0000 - tp: 39460.0000 - val_accuracy: 0.8123 - val_auc: 0.8560 - val_fn: 1743.0000 - val_fp: 1743.0000 - val_loss: 0.6163 - val_precision: 0.8123 - val_recall: 0.8123 - val_tn: 7542.0000 - val_tp: 7542.0000
Epoch 2/3
1462/1462 ━━━━━━━━━━━━━━━━━━━━ 1177s 805ms/step - accuracy: 0.9587 - auc: 0.9893 - fn: 1929.0000 - fp: 1929.0000 - loss: 0.1071 - precision: 0.9587 - recall: 0.9587 - tn: 44827.0000 - tp: 44827.0000 - val_accuracy: 0.9011 - val_auc: 0.9249 - val_fn: 918.0000 - val_fp: 918.0000 - val_loss: 0.4604 - val_precision: 0.9011 - val_recall: 0.9011 - val_tn: 8367.0000 - val_tp: 8367.0000
Epoch 3/3
1462/1462 ━━━━━━━━━━━━━━━━━━━━ 1169s 800ms/step - accuracy: 0.9788 - auc: 0.9950 - fn: 993.0000 - fp: 993.0000 - loss: 0.0580 - precision: 0.9788 - recall: 0.9788 - tn: 45763.0000 - tp: 45763.0000 - val_a

In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

val_predictions = model.predict(valid_X)
val_predictions_classes = np.argmax(val_predictions, axis=1)
val_true_classes = np.argmax(valid_Y, axis=1)

precision = precision_score(val_true_classes, val_predictions_classes, average='weighted')
recall = recall_score(val_true_classes, val_predictions_classes, average='weighted')
f1 = f1_score(val_true_classes, val_predictions_classes, average='weighted')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

291/291 ━━━━━━━━━━━━━━━━━━━━ 56s 193ms/step
Precision: 0.9100
Recall: 0.9068
F1-Score: 0.9077
